# 020. LSTM/GRU input/output shape

- return_sequences = False, True 일 때의 output 비교

- return_state = False, True 일 때의 internal state output 비교

- Bidirectional LSTM/GRU 의 output 비교

In [1]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Bidirectional, GRU
import numpy as np

T = 4   #Time Steps
D = 2   #features
U = 3   #LSTM units

X = np.random.randn(2, T, D)
print(X.shape)
X

(2, 4, 2)


array([[[-0.60449668, -1.24664934],
        [ 0.40303946,  1.07101827],
        [ 0.29868876, -0.26638805],
        [-0.93175437,  0.03344568]],

       [[-0.91455087,  0.2933444 ],
        [ 0.45004969,  0.664239  ],
        [ 0.70753989, -1.84704381],
        [-0.28847959,  0.97353427]]])

# LSTM

## return_sequences

- False (default) - last time step 의 output 만 반환
- True - 모든 timestep 의 output 을 모두 반환

In [2]:
def lstm(return_sequences=False):
    input_ = Input(shape=(T, D))
    rnn = LSTM(U, return_sequences=return_sequences)
    x = rnn(input_)
    
    model = Model(inputs=input_, outputs=x)

    return model.predict(X)

print("---- return_sequences=False ----> last timestep 의 output only")
lstm_out = lstm(return_sequences=False)
print(lstm_out.shape)
print(lstm_out)
print("\n---- return_sequences=True ----> 모든 timestep 별 output 출력")
lstm_out = lstm(return_sequences=True)
print(lstm_out.shape)
print(lstm_out)

---- return_sequences=False ----> last timestep 의 output only
(2, 3)
[[ 0.00513357 -0.10255466  0.03302112]
 [-0.01690003  0.03100773  0.04488242]]

---- return_sequences=True ----> 모든 timestep 별 output 출력
(2, 4, 3)
[[[-0.02614861 -0.10706683  0.249899  ]
  [-0.02930684  0.03175497 -0.03506815]
  [ 0.0290124  -0.012337   -0.03212378]
  [-0.15442696  0.05699279  0.04811271]]

 [[-0.18735111  0.09076463  0.04141849]
  [-0.09819879  0.04330634 -0.07939643]
  [ 0.00356378 -0.1048456   0.1835713 ]
  [-0.1236344   0.05416174  0.00731888]]]


## return_state

- False (default) - output 만 반환

- True - output, last step 의 hidden state, cell state (LSTM 의 경우) 반환

In [3]:
def lstm(return_state=False):
    input_ = Input(shape=(T, D))
    rnn = LSTM(U, return_state=return_state)
    x = rnn(input_)
    
    model = Model(inputs=input_, outputs=x)
    
    if return_state:
        o, h, c = model.predict(X)
        print("o :", o, o.shape)
        print("h :", h, h.shape)
        print("c :", c, c.shape)
    else:
        o = model.predict(X)
        print("o :", o, o.shape)

print("---- return_state=False ----> outout only")       
lstm(return_state=False)
print("\n---- return_state=True ----> outout, hidden state, cell state all")      
lstm(return_state=True)

---- return_state=False ----> outout only
o : [[-0.04347342  0.02627654  0.04072331]
 [ 0.03257031 -0.01137011 -0.08985841]] (2, 3)

---- return_state=True ----> outout, hidden state, cell state all
o : [[-0.04228009 -0.03854853  0.03806769]
 [-0.01475153 -0.0073417   0.09494869]] (2, 3)
h : [[-0.04228009 -0.03854853  0.03806769]
 [-0.01475153 -0.0073417   0.09494869]] (2, 3)
c : [[-0.0977716  -0.10251052  0.08563036]
 [-0.03258086 -0.02196478  0.21779428]] (2, 3)


# Bidirectional LSTM

- 순방향, 역방향이 concatenate 된 output 출력  

- hidden state, cell state 는 순방향, 역방향 별도 출력

In [4]:
T, D, U

(4, 2, 3)

In [5]:
def lstm(return_sequences=False, return_state=False):
    input_ = Input(shape=(T, D))
    rnn = Bidirectional(LSTM(U, return_state=return_state, return_sequences=return_sequences))
    x = rnn(input_)
    
    model = Model(inputs=input_, outputs=x)
    
    if return_state:    
        o, h1, c1, h2, c2 = model.predict(X)
        print("o :", o, o.shape)
        print("h1 :", h1, h1.shape)
        print("c1 :", c1, c1.shape)
        print("h2 :", h2, h2.shape)
        print("c2 :", c2, c2.shape)
    else:
        o = model.predict(X)
        print("o :", o, o.shape)

print("---- return_sequences = False ----> 순방향, 역방향이 concatenate 된 output")
lstm()
print()
print("---- return_sequences = True ----")
lstm(return_sequences=True)
print()
print("---- return_state = True ----")
lstm(return_state=True)

---- return_sequences = False ----> 순방향, 역방향이 concatenate 된 output
o : [[-0.02061366  0.01293562 -0.06001907  0.04084625  0.12373977 -0.03043511]
 [-0.05557535  0.0502746  -0.0132293   0.03855576 -0.00810303 -0.05197143]] (2, 6)

---- return_sequences = True ----
o : [[[-8.26302618e-02  7.37457424e-02 -1.35037228e-01 -6.03115819e-02
   -1.06566608e-01  2.08170131e-01]
  [-1.99066405e-03  1.99844148e-02  6.18372709e-02  7.02278912e-02
    1.13814242e-01 -3.48863043e-02]
  [ 3.36545520e-02  6.84683071e-03  1.51966298e-02  7.92219490e-03
   -8.96949247e-02  5.69425970e-02]
  [-4.41580303e-02  4.76020016e-02 -1.08828805e-04  4.29221205e-02
   -1.20062880e-01  1.08520031e-01]]

 [[-6.25013933e-02  2.89175846e-02  1.76630691e-02  5.08818179e-02
   -2.46921033e-02  4.43973131e-02]
  [ 2.02194750e-02 -3.48096453e-02  8.77216384e-02 -1.04589276e-02
    1.07743464e-01 -4.45611551e-02]
  [ 9.81246755e-02  3.95012796e-02 -8.78765211e-02 -1.63224533e-01
   -6.03674352e-02  8.33404064e-02]
  [ 4.757

# GRU 

- cell state 가 없는 것만 LSTM 과 차이

In [6]:
def gru(return_sequences=False, return_state=False):
    input_ = Input(shape=(T, D))
    rnn = GRU(U, return_state=return_state, return_sequences=return_sequences)
    x = rnn(input_)
    
    model = Model(inputs=input_, outputs=x)
    
    if return_state:    
        o, h = model.predict(X)
        print("o :", o, o.shape)
        print("h :", h, h.shape)
    else:
        o = model.predict(X)
        print("o :", o, o.shape)

print("---- return_sequences = False ----")
gru()
print()
print("---- return_sequences = True ----")
gru(return_sequences=True)
print()
print("---- return_state = True ----")
gru(return_state=True)

---- return_sequences = False ----
o : [[-0.2026882  -0.23462258 -0.02092821]
 [ 0.01406151  0.05019888 -0.35828817]] (2, 3)

---- return_sequences = True ----
o : [[[ 2.78185666e-01  5.33077419e-02  2.61873156e-01]
  [-4.16356474e-02  4.17959541e-02 -2.95471370e-01]
  [ 1.08707706e-02 -3.31743434e-02 -1.76400647e-01]
  [ 2.10270032e-01  5.05615883e-02  1.58388183e-01]]

 [[ 1.47782013e-01  8.61279070e-02  2.25567490e-01]
  [-8.61442611e-02  4.33125161e-02 -2.42092922e-01]
  [ 1.21110976e-01 -6.19138330e-02 -3.09961513e-02]
  [-2.10788246e-04  1.95841398e-03 -1.14017688e-01]]] (2, 4, 3)

---- return_state = True ----
o : [[ 0.04761061  0.1759149  -0.00727895]
 [-0.12537028 -0.1405216   0.12740481]] (2, 3)
h : [[ 0.04761061  0.1759149  -0.00727895]
 [-0.12537028 -0.1405216   0.12740481]] (2, 3)


# Bidirectional GRU

- cell state 가 없는 것 외에 LSTM 과 동일

In [7]:
def gru(return_sequences=False, return_state=False):
    input_ = Input(shape=(T, D))
    rnn = Bidirectional(GRU(U, return_state=return_state, return_sequences=return_sequences))
    x = rnn(input_)
    
    model = Model(inputs=input_, outputs=x)
    if return_state:    
        o, h1, h2 = model.predict(X)
        print("o :", o, o.shape)
        print("h1 :", h1, h1.shape)
        print("h2 :", h2, h2.shape)
    else:
        o = model.predict(X)
        print("o :", o, o.shape)

print("---- return_sequences = False ----")
gru(return_sequences=False)
print()
print("---- return_sequences = True ----")
gru(return_sequences=True)
print()
print("---- return_state = True ----")
gru(return_state=True)

---- return_sequences = False ----
o : [[ 0.04411835  0.02482697  0.20652452  0.01607268  0.25315988  0.02925531]
 [ 0.12634902  0.05026283  0.03065949 -0.15598059  0.19642803  0.01705931]] (2, 6)

---- return_sequences = True ----
o : [[[ 0.29100975 -0.02649402  0.6195369  -0.4958848   0.21773069
    0.19523914]
  [-0.10398487 -0.0520881   0.18867874  0.07622459 -0.13484493
   -0.20622937]
  [-0.10698281 -0.05301855  0.13188967 -0.11862679  0.01733194
    0.08266106]
  [ 0.03607223  0.03694398  0.32637125 -0.21733819 -0.00468401
   -0.1005182 ]]

 [[ 0.09138731  0.07021433  0.15198997 -0.18131296  0.01866308
   -0.14428364]
  [-0.1690068   0.01961014 -0.1214256  -0.09317569  0.09663606
    0.049572  ]
  [ 0.21044639 -0.02730048  0.45952764 -0.32058093  0.479533
    0.31322056]
  [ 0.02621562 -0.01071121  0.10469016  0.12911242 -0.09917925
   -0.28566483]]] (2, 4, 6)

---- return_state = True ----
o : [[-0.09450858 -0.2324524   0.11685339 -0.09827162  0.01246244  0.2099444 ]
 [-0.06184